# VascuLens AI — Clinical CAD and Vessel Model Training

**Multimodal AI Hackathon 2026 · Track A**

This notebook trains separate classifiers for overall CAD (`Cath` or `CAD`) and vessel-specific labels (`LAD`, `LCX`, `RCA`) from the UCI Extension of Z-Alizadeh Sani dataset. It excludes `Cath`, `CAD`, `LAD`, `LCX`, and `RCA` from every input feature matrix.

> Research/education only. This is not a diagnostic system. No 95% accuracy target is guaranteed. The 303-row dataset is small; report only measured metrics and validate externally.

Attach the original workbook/CSV as a Kaggle input before running the notebook. The original dataset is intentionally not included in Git.

Dataset: https://archive.ics.uci.edu/dataset/411/extention%2Bof%2Bz%2Balizadeh%2Bsani%2Bdataset

In [ ]:
import os, re, json, warnings, joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix
from sklearn.inspection import permutation_importance
SEED = 42
np.random.seed(SEED)
warnings.filterwarnings('ignore')


In [ ]:
roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path('.')]
files = []
for root in roots:
    if root.exists():
        files.extend([p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in {'.xlsx', '.xls', '.csv'}])
files = sorted(set(files))
for i, path in enumerate(files[:40]): print(i, path)
preferred = [p for p in files if any(k in p.name.lower() for k in ['z-alizadeh', 'z_alizadeh', 'extention', 'alizadeh sani'])]
if not preferred:
    raise FileNotFoundError('Attach the original UCI Extension of Z-Alizadeh Sani XLSX/CSV as a Kaggle input and rerun.')
DATA_PATH = preferred[0]
raw = pd.read_csv(DATA_PATH) if DATA_PATH.suffix.lower() == '.csv' else pd.read_excel(DATA_PATH, sheet_name=0)
raw.columns = [str(c).replace('\\ufeff','').strip() for c in raw.columns]
print('Dataset:', DATA_PATH, 'shape:', raw.shape)
display(raw.head())
display(pd.DataFrame({'column':raw.columns, 'dtype':[str(raw[c].dtype) for c in raw.columns], 'missing':[int(raw[c].isna().sum()) for c in raw.columns]}))


In [ ]:
def norm_col(x): return re.sub(r'[^a-z0-9]+', '', str(x).strip().lower())
def find_col(candidates):
    lookup = {norm_col(c):c for c in raw.columns}
    return next((lookup[norm_col(x)] for x in candidates if norm_col(x) in lookup), None)
def encode_target(s):
    z = s.astype(str).str.strip().str.lower()
    if set(z.unique()) <= {'0','1','0.0','1.0'}: return z.astype(float).astype(int)
    positive = {'cad','yes','positive','present','disease','stenosis','stenotic','true','abnormal'}
    negative = {'normal','no','negative','absent','healthy','false','non-cad','noncad','nonstenotic','non stenotic'}
    unseen = set(z.unique()) - positive - negative
    if unseen: raise ValueError(f'Unknown target labels {sorted(unseen)}. Add an explicit mapping; do not guess.')
    return z.isin(positive).astype(int)
target_cols = {'CAD':find_col(['CAD','Cath']), 'LAD':find_col(['LAD']), 'LCX':find_col(['LCX']), 'RCA':find_col(['RCA'])}
print('Target columns:', target_cols)
if target_cols['CAD'] is None: raise ValueError('Could not find Cath or CAD target.')
excluded = {'cath','cad','lad','lcx','rca','id','patientid'}
feature_cols = [c for c in raw.columns if norm_col(c) not in excluded and not norm_col(c).startswith('unnamed')]
X_all = raw[feature_cols].copy()
for c in X_all.columns:
    if X_all[c].dtype == 'object':
        parsed = pd.to_numeric(X_all[c], errors='coerce')
        valid = X_all[c].notna().sum()
        if valid and parsed.notna().sum()/valid >= .95: X_all[c] = parsed
assert not ({norm_col(c) for c in feature_cols} & excluded), 'Leakage guard failed.'
print('Rows:', len(raw), '| feature count:', len(feature_cols), '| excluded outcomes:', sorted(excluded))
display(pd.DataFrame({'target':list(target_cols), 'column':list(target_cols.values())}))


In [ ]:
def make_preprocessor(X):
    numeric = X.select_dtypes(include=['number','bool']).columns.tolist()
    categorical = [c for c in X.columns if c not in numeric]
    transformers=[]
    if numeric: transformers.append(('num', Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]), numeric))
    if categorical: transformers.append(('cat', Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(handle_unknown='ignore'))]), categorical))
    return ColumnTransformer(transformers, remainder='drop')
def candidates():
    return {
      'LogisticRegression': LogisticRegression(max_iter=3000,class_weight='balanced',C=.5,random_state=SEED),
      'RBF_SVM': SVC(C=1.0,kernel='rbf',probability=True,class_weight='balanced',random_state=SEED),
      'ExtraTrees': ExtraTreesClassifier(n_estimators=300,min_samples_leaf=2,max_features='sqrt',class_weight='balanced',random_state=SEED,n_jobs=1),
      'RandomForest': RandomForestClassifier(n_estimators=300,min_samples_leaf=2,max_features='sqrt',class_weight='balanced',random_state=SEED,n_jobs=1)
    }
def metric_pack(y_true, prob, threshold):
    pred=(prob>=threshold).astype(int); tn,fp,fn,tp=confusion_matrix(y_true,pred,labels=[0,1]).ravel()
    return {'threshold':float(threshold),'accuracy':accuracy_score(y_true,pred),'balanced_accuracy':balanced_accuracy_score(y_true,pred),'precision':precision_score(y_true,pred,zero_division=0),'sensitivity_recall':recall_score(y_true,pred,zero_division=0),'specificity':tn/(tn+fp) if tn+fp else np.nan,'f1':f1_score(y_true,pred,zero_division=0),'roc_auc':roc_auc_score(y_true,prob) if len(np.unique(y_true))==2 else np.nan,'average_precision':average_precision_score(y_true,prob),'brier_score':brier_score_loss(y_true,prob),'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp)}


In [ ]:
results=[]; models={}; defaults={}
for c in feature_cols:
    s=X_all[c]
    if pd.api.types.is_numeric_dtype(s): defaults[c]=float(s.median()) if s.notna().any() else 0.0
    else:
        mode=s.dropna().mode(); defaults[c]=str(mode.iloc[0]) if not mode.empty else ''
for target, col in target_cols.items():
    if col is None: print('Skip missing target:', target); continue
    mask=raw[col].notna(); X=X_all.loc[mask].reset_index(drop=True); y=encode_target(raw.loc[mask,col]).reset_index(drop=True)
    counts=y.value_counts(); print('\\nTarget',target,'counts',counts.to_dict())
    if y.nunique()!=2 or int(counts.min())<5: print('Skipping: insufficient class counts'); continue
    X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=SEED)
    cv=StratifiedKFold(n_splits=min(5,int(y_train.value_counts().min())),shuffle=True,random_state=SEED)
    scores=[]
    for name, estimator in candidates().items():
        pipe=Pipeline([('preprocessor',make_preprocessor(X_train)),('classifier',estimator)])
        try:
            vals=cross_val_score(pipe,X_train,y_train,cv=cv,scoring='roc_auc',n_jobs=1)
            scores.append((float(vals.mean()),float(vals.std()),name,pipe))
        except Exception as e: print('Model failed',name,repr(e))
    if not scores: print('No candidates passed for',target); continue
    cv_auc,cv_sd,name,model=max(scores,key=lambda x:x[0])
    oof=cross_val_predict(model,X_train,y_train,cv=cv,method='predict_proba',n_jobs=1)[:,1]
    grid=np.linspace(.2,.8,61); threshold=max(((balanced_accuracy_score(y_train,oof>=t),t) for t in grid),key=lambda x:x[0])[1]
    model.fit(X_train,y_train); prob=model.predict_proba(X_test)[:,1]; metrics=metric_pack(y_test,prob,threshold)
    try:
        pi=permutation_importance(model,X_test,y_test,n_repeats=4,random_state=SEED,scoring='roc_auc',n_jobs=1)
        imp=sorted([{'feature':str(f),'mean_auc_drop':float(v)} for f,v in zip(X_test.columns,pi.importances_mean)],key=lambda z:z['mean_auc_drop'],reverse=True)[:15]
    except Exception as e: imp=[]; print('Importance failed:',repr(e))
    results.append({'target':target,'target_column':col,'selected_model':name,'training_cv_roc_auc_mean':cv_auc,'training_cv_roc_auc_sd':cv_sd,'train_n':len(y_train),'holdout_n':len(y_test),**metrics})
    models[target]=model
    print('Selected:',name,'training CV ROC-AUC:',round(cv_auc,3),'holdout:',{k:round(v,3) for k,v in metrics.items() if isinstance(v,(float,np.floating))})
    print('Top global permutation importance:',imp[:5])
if not results: raise RuntimeError('No targets were trained. Verify source labels and class counts.')
results_df=pd.DataFrame(results); display(results_df.round(4))


In [ ]:
fig,ax=plt.subplots(figsize=(10,4))
ax.bar(results_df['target'],results_df['accuracy'],label='Accuracy')
ax.scatter(results_df['target'],results_df['roc_auc'],label='ROC-AUC',color='crimson',zorder=3)
ax.set_ylim(0,1); ax.set_title('Held-out metrics by target'); ax.legend(); ax.grid(axis='y',alpha=.2); plt.show()
OUT=Path('/kaggle/working/vasculens_artifacts'); OUT.mkdir(parents=True,exist_ok=True)
joblib.dump({'models':models,'features':feature_cols,'defaults':defaults},OUT/'clinical.joblib')
results_df.to_csv(OUT/'holdout_metrics.csv',index=False)
summary={'project':'VascuLens AI','dataset':str(DATA_PATH.name),'rows':int(len(raw)),'features':feature_cols,'excluded_targets':['Cath','CAD','LAD','LCX','RCA'],'seed':SEED,'validation':'stratified 80/20 holdout, training-only CV model selection','metrics':results_df.replace({np.nan:None}).to_dict(orient='records'),'note':'Small dataset; no clinical validation. Do not claim a target accuracy unless observed.'}
(OUT/'metrics.json').write_text(json.dumps(summary,indent=2,allow_nan=False))
print('Artifacts saved to',OUT)
print('Metrics below are measured only for the attached dataset and this fixed holdout:')
display(results_df.round(4))


## Reporting and clinical safety

Use the metrics printed above, not previous or illustrative numbers. The best model is selected by training-fold ROC-AUC; the holdout is evaluated once for each target. A single split on 303 records is noisy, and this is not external validation. Do not promise 95% accuracy or use these scores for clinical decisions.

The clinical data are tabular and do not supply lesion coordinates or patient-matched CT volumes. The ImageCAS segmentation pathway is separate until paired data and compatible labels are available.